# Week 4: Design Patterns for ML
## Strategy for models, Observer for training callbacks, Factory for dataset loaders, pytest

**Course:** AI for Aerospace & Mechanical Engineering (PhD) &nbsp;|&nbsp; **Block 0:** Python & OOP crash course

> **Provenance:** new content. The instructor's PDFs do not cover software design patterns.
> Pattern definitions are rewritten (not copied) from GeeksforGeeks ("Strategy Design Pattern",
> "Observer Design Pattern", "Factory Method - Python Design Patterns", "Getting Started with
> Pytest", "Pytest Fixtures") and from Gamma, Helm, Johnson and Vlissides, *Design Patterns*
> (1994); Python details follow the W3Schools tutorial (Classes, Inheritance, Polymorphism,
> Dictionaries, Lambda). The KodAmoz Python bootcamp (kodamoz.com/fa/bootcamps/python) was
> checked for Persian terminology; it covers OOP but not design patterns or testing. Least
> squares and k-NN follow Bishop, *PRML* (2006), Ch. 3 and Sec. 2.5.2; the fuel-flow model
> is adapted from Anderson, *Aircraft Performance and Design*, Ch. 5.

**How to use this notebook.** Run it top to bottom. Cells marked `# TODO` are your exercises;
each is followed by a check that prints `[x]` when your answer is right and `[ ]` while it is
not done. The solved copy is `week04_design_patterns_solved.ipynb`.

## 0. The problem: one experiment script that keeps growing

We want to predict the **fuel flow** of a small piston UAV (g/h) from airspeed, altitude,
climb rate and mass. This is the Week 6 regression problem; this week we build the software
that Week 6 to Week 13 will reuse. A typical first attempt is one function with flags:

* `model_type` chooses the model with an `if/elif` chain,
* `data_source` chooses the loader with another chain,
* `verbose`, `early_stop`, `patience` add logging and stopping *inside* the training loop.

Every new model, data source or training feature means editing this one function. Run the
cell and look at what goes wrong.

In [ ]:
from __future__ import annotations

import math
import subprocess
import sys
from collections.abc import Callable
from pathlib import Path
from typing import ClassVar, Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

FUEL_CSV: Final[Path] = REPO_ROOT / "data" / "sample" / "week04_fuel_flow.csv"
G0: Final[float] = 9.80665          # standard gravity [m/s^2]
SEED: Final[int] = 4                 # seed of the course fuel-flow data


def check(name: str, value: object, expected: float, tol: float) -> None:
    """Print a pass mark if ``value`` is within ``tol`` of ``expected``.

    A value of None means the exercise has not been attempted yet.
    """
    if value is None:
        print(f"[ ] {name}: not done yet")
        return
    assert abs(float(value) - expected) <= tol, f"{name}: got {value}, expected ~{expected}"
    print(f"[x] {name} = {float(value):.6g}")


print("Repository root found:", REPO_ROOT.name)

In [ ]:
def run_experiment(model_type: str, data_source: str, verbose: bool = False,
                   early_stop: bool = False, patience: int = 5) -> float:
    """The 'one big function' version (deliberately bad style)."""
    if data_source == "csv":
        frame = pd.read_csv(FUEL_CSV)
    elif data_source == "synthetic":
        frame = pd.read_csv(FUEL_CSV).sample(frac=1.0, random_state=0)
    else:
        frame = pd.read_csv(FUEL_CSV)                 # silent fallback
    X = frame.iloc[:, :4].to_numpy()
    y = frame.iloc[:, 4].to_numpy()
    n_train = int(0.75 * len(y))
    if model_type == "linear":
        A = np.column_stack([np.ones(n_train), X[:n_train]])
        w = np.linalg.lstsq(A, y[:n_train], rcond=None)[0]
        pred = np.column_stack([np.ones(len(y) - n_train), X[n_train:]]) @ w
    elif model_type == "knn":
        d = ((X[n_train:, None, :] - X[None, :n_train, :]) ** 2).sum(axis=2)
        pred = y[:n_train][np.argsort(d, axis=1)[:, :5]].mean(axis=1)
    else:
        pred = np.full(len(y) - n_train, y[:n_train].mean())   # silent fallback
    # ... imagine 40 more lines here for gradient descent with verbose/early_stop flags
    return float(np.sqrt(np.mean((y[n_train:] - pred) ** 2)))


for model_type in ["linear", "knn", "KNN", "lienar"]:
    print(f"{model_type:<8} test RMSE = {run_experiment(model_type, 'csv'):7.2f} g/h")

**What went wrong?**

1. `"KNN"` and `"lienar"` fall through to the `else`: we get the **mean baseline** without any
   error, and could report it as a k-NN result.
2. The k-NN branch does not scale the features, so altitude (thousands of metres) dominates the
   distance; fixing it means editing this function again.
3. Adding a model, a data source or a training feature (logging, early stopping, a learning-rate
   schedule) always edits the *same* function: every change risks breaking the other branches,
   and nothing can be tested in isolation.

Three classic design patterns (Gamma et al., 1994) fix the three axes of change:

| What varies | Pattern | Course class |
|-------------|---------|--------------|
| the **model** (algorithm) | Strategy | `Regressor` + `RegressionExperiment` |
| what happens **during training** | Observer | `Trainer` + `Callback` |
| where the **data** comes from | Factory | `DatasetFactory` + `DatasetLoader` |

First, the engineering behind the data.

### Worked Example 1 (by hand, then code): fuel flow at 25 m/s

Power required = parasite + induced + climb power, then shaft power = $P_\text{req}/\eta_p$ and
fuel flow = BSFC × shaft power. For $V = 25$ m/s at sea level ($\rho = 1.225$ kg/m³), level
flight, $m = 30$ kg ($W = 294.20$ N), $S = 1.5$ m², $C_{D0} = 0.03$, $AR = 10$, $e = 0.8$,
$\eta_p = 0.75$, BSFC = 0.35 kg/kWh:

$$P_\text{par} = \tfrac12 \rho V^3 S C_{D0} = 0.5 \times 1.225 \times 15\,625 \times 1.5 \times 0.03 = 430.66\ \text{W}$$
$$P_\text{ind} = \frac{2W^2}{\rho V S \pi e AR} = \frac{2 \times 294.20^2}{1.225 \times 25 \times 1.5 \times \pi \times 0.8 \times 10} = \frac{173\,106}{1154.5} = 149.94\ \text{W}$$
$$P_\text{shaft} = (430.66 + 149.94)/0.75 = 774.13\ \text{W}, \qquad \dot m_f = 0.35 \times 0.77413\ \text{kW} = 0.27095\ \text{kg/h} = 270.95\ \text{g/h}.$$

In [ ]:
from aerospace_ai.datasets import FuelFlowAircraft

uav = FuelFlowAircraft()
fuel_25 = float(uav.fuel_flow_gph(25.0, 0.0, 0.0, 30.0))
print(f"model: {fuel_25:.2f} g/h")
check("Worked Example 1 fuel flow", fuel_25, 270.95, 0.01)

**Exercise 0.1.** Repeat the hand calculation for a **1.0 m/s climb** at the same speed and mass.
Climb power is $W \cdot \dot h = 294.20$ W, so shaft power grows by $294.20/0.75 = 392.3$ W.
Predict the fuel flow by hand first, then compute it with `uav.fuel_flow_gph`.

In [ ]:
fuel_climb = None      # TODO: uav.fuel_flow_gph(...) for V = 25 m/s, h = 0 m, climb = 1 m/s, 30 kg
check("Exercise 0.1 climbing fuel flow", fuel_climb, 408.24, 0.01)

## 1. Strategy: make the model a replaceable part

**Idea** (GeeksforGeeks, "Strategy Design Pattern"): put each algorithm in its own class behind a
common interface; the code that *uses* the algorithm (the **context**) holds a reference to "some
strategy" and never asks which one. Swapping the algorithm is one assignment, and adding one is
one new class: the **Open/Closed Principle** (open for extension, closed for modification).

### 1.1 The lightest strategy in Python: a function

Functions are objects in Python (W3Schools, "Python Lambda"), so a dictionary of functions is
already a strategy table. Unknown names now fail loudly with a `KeyError`.

In [ ]:
Predictor = Callable[[np.ndarray, np.ndarray, np.ndarray], np.ndarray]

PREDICTORS: dict[str, Predictor] = {
    "mean": lambda X_tr, y_tr, X_te: np.full(len(X_te), y_tr.mean()),
    "median": lambda X_tr, y_tr, X_te: np.full(len(X_te), np.median(y_tr)),
}

y_train_demo = np.array([170.0, 200.0, 280.0, 400.0])
for name in ["mean", "median"]:
    print(name, PREDICTORS[name](None, y_train_demo, np.zeros(2)))
try:
    PREDICTORS["KNN"]
except KeyError as err:
    print("KeyError:", err, "<- the typo is caught")

A function is enough while a strategy has **no state and no settings**. Real models have both:
learned parameters (`coef_`) and hyperparameters (`k`, `alpha`), and they need the same input
checks every time. That is what a class hierarchy gives us.

### 1.2 The course strategy interface: `Regressor`

`aerospace_ai.models.Regressor` is an abstract base class. Its public `fit` and `predict` are
*template methods* (as `BaseSensor.add_reading` was in Week 2): they check shapes, finiteness
and the fit status once for every model, then call the abstract hooks `_fit` and `_predict`
that each strategy implements.

In [ ]:
import inspect

from aerospace_ai.models import (
    GradientDescentRegressor,
    KNearestNeighbors,
    LinearLeastSquares,
    MeanBaseline,
    NotFittedError,
    RegressionExperiment,
    Regressor,
    SklearnRegressor,
    rmse,
)

print(inspect.getsource(MeanBaseline))
try:
    MeanBaseline().predict(np.zeros((1, 4)))
except NotFittedError as err:
    print("NotFittedError:", err)

### Worked Example 2 (by hand, then code): three strategies, four points

Fuel flow at sea level vs airspeed only (one feature): training points
$(18, 170), (22, 200), (26, 280), (30, 400)$; test points $(25, 270)$ and $(29, 375)$ g/h.

* **Mean baseline:** $\bar y = 262.5$ for both. Errors $-7.5$, $-112.5$;
  RMSE $= \sqrt{(7.5^2 + 112.5^2)/2} = 79.73$ g/h.
* **k-NN, k = 2 (no scaling):** neighbours of 25 are 26 and 22 → $(280+200)/2 = 240$; of 29 are
  30 and 26 → $340$. Errors $-30$, $-35$; RMSE $= \sqrt{(900 + 1225)/2} = 32.60$ g/h.
* **Least squares:** $\bar x = 24$, $\bar y = 262.5$, $S_{xy} = \sum (x-\bar x)(y-\bar y) = 1540$,
  $S_{xx} = 80$, slope $= 19.25$, intercept $= 262.5 - 19.25 \times 24 = -199.5$. Predictions
  $281.75$ and $358.75$; RMSE $= \sqrt{(11.75^2 + 16.25^2)/2} = 14.18$ g/h.

Inside the training range the straight line wins. At $V = 16$ m/s (true ≈ 162 g/h) it predicts
$108.5$ while k-NN predicts $185$: the best strategy depends on the question, so the code must
make swapping cheap.

In [ ]:
X4 = np.array([[18.0], [22.0], [26.0], [30.0]])
y4 = np.array([170.0, 200.0, 280.0, 400.0])
X_test4 = np.array([[25.0], [29.0]])
y_test4 = np.array([270.0, 375.0])

hand_rmse = {}
for model in [MeanBaseline(), KNearestNeighbors(k=2, standardize=False), LinearLeastSquares()]:
    model.fit(X4, y4)            # same two calls for every strategy: polymorphism
    hand_rmse[model.name] = model.score(X_test4, y_test4)
    print(f"{model!r:<45} predictions {model.predict(X_test4)}  RMSE {hand_rmse[model.name]:.2f}")

check("Worked Example 2 mean RMSE", hand_rmse["mean_baseline"], 79.73, 0.01)
check("Worked Example 2 k-NN RMSE", hand_rmse["k_nearest_neighbors"], 32.60, 0.01)
check("Worked Example 2 linear RMSE", hand_rmse["linear_least_squares"], 14.18, 0.01)

**Exercise 1.1.** Write a new strategy `MedianBaseline` (predict the training **median**; robust
to the fuel-flow outliers a clogged flow meter produces). Subclass `Regressor`, set the class
attribute `name = "median_baseline"`, store the median in `_fit` and return it in `_predict`.
You get input checks, `score` and `repr` for free. On the four training points the median is
$(200 + 280)/2 = 240$.

In [ ]:
class MedianBaseline(Regressor):
    """Always predict the training median."""

    name: ClassVar[str] = "median_baseline"

    def __init__(self) -> None:
        super().__init__()
        self.median_: float = 0.0

    def _fit(self, X: np.ndarray, y: np.ndarray) -> None:
        raise NotImplementedError   # TODO: store the median of y in self.median_

    def _predict(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError   # TODO: one value per row of X, all equal to self.median_


try:
    median_prediction = float(MedianBaseline().fit(X4, y4).predict(X_test4)[0])
except NotImplementedError:
    median_prediction = None
check("Exercise 1.1 median prediction", median_prediction, 240.0, 1e-9)

### 1.3 The context: `RegressionExperiment`

The experiment fits a strategy on the training set and scores it on the test set. It never
contains an `if` on the model type. We load the real 400-row table through the factory (Section
3 explains how) and use a **chronological** split: train on the first 75 %, test on the last
25 %, the leak-free default you will meet again in Week 5.

In [ ]:
from aerospace_ai.datasets import DatasetFactory

fuel = DatasetFactory.load("fuel_flow_synthetic", seed=SEED)
train, test = fuel.split(test_fraction=0.25)
print(fuel, "\n", train, "\n", test)
display(fuel.to_frame().describe().round(2))

experiment = RegressionExperiment(MeanBaseline())
strategies: list[Regressor] = [
    MeanBaseline(),
    LinearLeastSquares(),
    LinearLeastSquares(alpha=10.0),
    KNearestNeighbors(k=5),
    KNearestNeighbors(k=5, standardize=False),
    GradientDescentRegressor(learning_rate=0.1, epochs=200),
]
results = experiment.compare(strategies, train, test)
table = pd.DataFrame(results).set_index("model").round(3)
table

Note the unscaled k-NN: it is **worse than predicting the mean** (test RMSE about 235 vs 225
g/h), because altitude (0 to 3000 m) swamps airspeed (16 to 36 m/s) in the distance, so the
"nearest" flights are simply those at a similar altitude. The same algorithm with a different
setting is a different strategy, and a side-by-side comparison exposes the scaling bug that was
buried inside `run_experiment`.
`GradientDescentRegressor` and `LinearLeastSquares` reach the same answer by different routes
(iterative vs closed form), which is a useful self-check.

**Exercise 1.2 (Adapter).** scikit-learn estimators already have `fit`/`predict`, but they are
not `Regressor` subclasses. `SklearnRegressor` is an **adapter**: it wraps any object with
`fit`/`predict` (checked structurally with a `typing.Protocol`) so it can join the comparison.
Add `sklearn.ensemble.RandomForestRegressor(n_estimators=200, random_state=0)` to the
comparison and store its test RMSE in `forest_rmse`. It should beat every linear model,
because a forest captures the curvature and the climb × mass interaction.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

FOREST_RMSE: Final[float] = 37.65   # expected test RMSE with scikit-learn 1.4 [g/h]

forest = None          # TODO: SklearnRegressor(RandomForestRegressor(n_estimators=200, random_state=0))
forest_rmse = None     # TODO: experiment.strategy = forest; experiment.run(train, test).test_rmse
check("Exercise 1.2 forest test RMSE", forest_rmse, FOREST_RMSE, 1.0)

## 2. Observer: training events without touching the loop

A training loop has many interested parties: a **history** recorder for plots, a **logger**,
an **early-stopping** rule, a **divergence guard**, a **learning-rate schedule**, later a
checkpoint writer. Putting all of them in the loop gives the `verbose=..., early_stop=...` flag
soup of Section 0.

**Idea** (GeeksforGeeks, "Observer Design Pattern"): a **subject** keeps a list of
**observers** and notifies all of them when something happens; it does not know what they do.
Here the subject is `Trainer`, the events are `on_train_begin`, `on_epoch_end` and
`on_train_end`, and observers are `Callback` objects. Observers talk back only through the
shared `TrainingState` (`state.request_stop(...)`, `state.learning_rate = ...`).

### Worked Example 3 (by hand, then code): one gradient-descent step

Model $\hat y = w_0 + w_1 x$, data $(1, 2), (2, 4), (3, 5)$, start $w = (0, 0)$, $\eta = 0.1$,
loss MSE $L = \frac1n \sum (\hat y_i - y_i)^2$.

* Residuals $\hat y - y = (-2, -4, -5)$, so $L_0 = (4 + 16 + 25)/3 = 15$.
* Gradient $\nabla L = \frac{2}{n} A^\top(Aw - y)$ with rows $A_i = (1, x_i)$:
  $\partial L/\partial w_0 = \frac23(-11) = -7.333$,
  $\partial L/\partial w_1 = \frac23(-2 - 8 - 15) = -16.667$.
* Step: $w \leftarrow w - \eta \nabla L = (0.7333, 1.6667)$.
* New predictions $(2.4, 4.0667, 5.7333)$, residuals $(0.4, 0.0667, 0.7333)$, so
  $L_1 = (0.16 + 0.0044 + 0.5378)/3 = 0.2341$. One step cut the loss by a factor of 64.

In [ ]:
X3 = np.array([[1.0], [2.0], [3.0]])
y3 = np.array([2.0, 4.0, 5.0])
gd = GradientDescentRegressor(learning_rate=0.1, epochs=1, standardize=False)
gd.initialize(X3, y3)
print("initial loss:", gd.loss(X3, y3))
print("gradient:    ", gd.gradient(X3, y3))
loss_after = gd.train_step(X3, y3)
print("weights:     ", gd.weights_)
print("residuals:   ", gd.predict(X3) - y3)
print("loss after:  ", loss_after)
check("Worked Example 3 w1 after one step", gd.weights_[1], 1.6667, 1e-4)
check("Worked Example 3 loss after one step", loss_after, 0.2341, 1e-4)

### 2.1 The subject: `Trainer` and its observers

`Trainer.fit` initialises the model, then for each epoch: one `train_step`, the validation loss,
`notify("on_epoch_end")`, and a check of `state.stop_training`. That is the whole loop: about
fifteen lines that will not change when we add new behaviour. We train on the first 75 % of
the training set and validate on the rest (still chronological).

In [ ]:
from aerospace_ai.training import (
    Callback,
    DivergenceGuard,
    EarlyStopping,
    History,
    ProgressLogger,
    StepDecay,
    Trainer,
    TrainingState,
)

fit_set, val_set = train.split(test_fraction=0.25)
print(fit_set, "\n", val_set)

history = History()
trainer = Trainer([history, ProgressLogger(every=25)])
state = trainer.fit(GradientDescentRegressor(learning_rate=0.1), fit_set.X, fit_set.y,
                    val_set.X, val_set.y, max_epochs=100)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogy(np.sqrt(history.train_loss), label="train RMSE")
ax.semilogy(np.sqrt(history.val_loss), label="validation RMSE")
ax.set_xlabel("epoch")
ax.set_ylabel("RMSE [g/h]")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

### 2.2 Early stopping and the divergence guard

`EarlyStopping(patience, min_delta)` counts epochs whose validation loss is **not** at least
`min_delta` below the best so far; after `patience` such epochs it requests a stop, and at train
end it restores the best weights. With `min_delta = 0` it waits for the loss to stop changing
in the last digit; a sensible `min_delta` (here 1 (g/h)², i.e. "less than one unit of MSE is
not progress") stops much earlier with practically the same model. In-class Problem 2 traces
the counter by hand.

`DivergenceGuard` is the training analogue of an over-temperature cut-off: if the loss becomes
non-finite or grows ten times above its first value, it stops the run. Too large a learning
rate (here 1.5) is the usual cause.

In [ ]:
runs = {}
for min_delta in (0.0, 1.0):
    stopper = EarlyStopping(patience=5, min_delta=min_delta)
    model = GradientDescentRegressor(learning_rate=0.1)
    st = Trainer([stopper]).fit(model, fit_set.X, fit_set.y, val_set.X, val_set.y, max_epochs=1000)
    runs[min_delta] = (st.epoch + 1, stopper.best_epoch, model.score(test.X, test.y))
    print(f"min_delta={min_delta:>4}: ran {st.epoch + 1:4d} epochs, best epoch {stopper.best_epoch:4d}, "
          f"test RMSE {runs[min_delta][2]:.3f} g/h")

guard = DivergenceGuard(factor=10.0)
st = Trainer([guard, ProgressLogger(every=1)]).fit(
    GradientDescentRegressor(learning_rate=1.5), fit_set.X, fit_set.y, max_epochs=100
)
print("guard triggered:", guard.triggered)

**Why the learning rate 1.5 diverges.** With standardised features the MSE Hessian is
$H = \frac{2}{n}A^\top A$ (rows $A_i = (1, \tilde x_i)$). For our training set its eigenvalues
are 1.85 to 2.17. Gradient descent on a quadratic is stable only if $\eta < 2/\lambda_\max$,
here $\eta < 2/2.17 = 0.92$, so $\eta = 1.5$ must diverge while $\eta = 0.1$ is safe.
Week 6 derives this properly; today the point is that a *separate observer* catches it.

**Exercise 2.1.** Write a callback `RmseTargetStop(target_rmse)` that stops training as soon as
the **validation RMSE** ($\sqrt{\text{val\_loss}}$) is at or below a target, with the reason
`"target RMSE reached"`. Engineering use: stop as soon as the model meets the accuracy
requirement in the specification. Override only `on_epoch_end`. With a 60 g/h target and the
settings below, training stops after a few epochs; store the number of epochs run
(`state.epoch + 1`) in `epochs_to_target`.

In [ ]:
TARGET_RMSE_GPH: Final[float] = 60.0
EPOCHS_TO_TARGET: Final[int] = 12     # expected answer


class RmseTargetStop(Callback):
    """Stop when the validation RMSE reaches a target value."""

    def __init__(self, target_rmse: float) -> None:
        if target_rmse <= 0.0:
            raise ValueError("target_rmse must be positive")
        self.target_rmse = target_rmse

    def on_epoch_end(self, state: TrainingState) -> None:
        pass   # TODO: if sqrt(state.val_loss) <= self.target_rmse: state.request_stop("target RMSE reached")


st = Trainer([RmseTargetStop(TARGET_RMSE_GPH), ProgressLogger(every=5)]).fit(
    GradientDescentRegressor(learning_rate=0.1), fit_set.X, fit_set.y,
    val_set.X, val_set.y, max_epochs=200,
)
epochs_to_target = st.epoch + 1 if st.stop_reason == "target RMSE reached" else None
check("Exercise 2.1 epochs to reach 60 g/h", epochs_to_target, EPOCHS_TO_TARGET, 0)

## 3. Factory: ask for data by name

The data source is the third axis of change: today a synthetic generator and a CSV file, next
week a noisy LiDAR log, in Week 11 a folder of camera images. Client code should say *what* it
wants (`"fuel_flow_synthetic"`) and not *how* to build it.

**Idea** (GeeksforGeeks, "Factory Method - Python Design Patterns"): move object creation out of
the client into a factory, so the client depends only on the abstract product
(`DatasetLoader`, which has one method: `load()`). The Pythonic form is a **registry**: a
class-level dictionary from name to class, filled by a class decorator. A new loader registers
itself; neither the factory nor any experiment code is edited.

In [ ]:
from aerospace_ai.datasets import (
    FUEL_FEATURES,
    FUEL_TARGET,
    CsvTableLoader,
    Dataset,
    DatasetLoader,
    UnknownDatasetError,
)

print("registered:", DatasetFactory.available())
loader = DatasetFactory.create("fuel_flow_synthetic", n_samples=5, seed=1)
print(loader, "->", loader.load())
try:
    DatasetFactory.create("fuel_flow_synthtic")
except UnknownDatasetError as err:
    print("UnknownDatasetError:", err)

Compare with Section 0: an unknown name is now an error *with a suggestion*, and the loader's
settings (`n_samples`, `seed`) are passed as keyword arguments instead of new function flags.

**Exercise 3.1.** Register a loader `"fuel_flow_csv"` for the CSV file
`data/sample/week04_fuel_flow.csv`. Subclass `CsvTableLoader` (which already reads a CSV and
picks columns) and give it a constructor with **defaults**: `path=FUEL_CSV`,
`features=FUEL_FEATURES`, `target=FUEL_TARGET`, passed to `super().__init__`. The CSV was written
from the synthetic loader with three decimals, so the two datasets must agree to 0.001 g/h.

Notebook detail: re-running a cell defines a *new* class object, and the registry refuses to
give an existing name to a different class (that protects you from silent overwrites). The
first line of the cell removes the old registration so the cell can be re-run.

In [ ]:
if "fuel_flow_csv" in DatasetFactory.available():
    DatasetFactory.unregister("fuel_flow_csv")      # allow re-running this cell

# TODO: decorate with @DatasetFactory.register("fuel_flow_csv") and complete __init__
class FuelFlowCsvLoader(CsvTableLoader):
    """The Week 4 fuel-flow CSV with its default columns."""

    def __init__(self, path: str | Path = FUEL_CSV) -> None:
        raise NotImplementedError   # TODO: super().__init__(path, FUEL_FEATURES, FUEL_TARGET)


try:
    from_csv = DatasetFactory.load("fuel_flow_csv")
    csv_mismatch = float(np.max(np.abs(from_csv.y - fuel.y)))
    print(from_csv)
except (UnknownDatasetError, NotImplementedError):
    csv_mismatch = None
check("Exercise 3.1 CSV vs synthetic max difference", csv_mismatch, 0.0, 1e-3)

## 4. Testing patterns with pytest

Patterns make testing *easier*, because each piece can be tested alone:

* **Contract tests for strategies.** One parametrized fixture runs the same tests on every
  `Regressor` (fit returns `self`, prediction shape, `NotFittedError`, bad data raises, beats
  the baseline). A new model added to the list is tested for free. This is the
  Liskov Substitution Principle turned into code: any strategy must be usable where a
  `Regressor` is expected.
* **Test doubles for observers.** A `SpyCallback` records the events it receives, so we can
  assert the exact notification order without any real model; a `ScriptedModel` returns a fixed
  list of validation losses, so the early-stopping trace of In-class Problem 2 becomes a test.
* **Isolation for factories.** A fixture registers a throw-away loader and `unregister`s it
  afterwards, so tests do not leak into each other; `tmp_path` gives a fresh folder for CSV files.

Useful pytest tools this week (GeeksforGeeks, "Pytest Fixtures"; pytest docs): `@pytest.fixture`
(also with `params=`), `@pytest.mark.parametrize`, `pytest.raises(..., match=...)`,
`pytest.approx`, the built-in fixtures `tmp_path` and `capsys`, and `pytest.importorskip`.
The cell runs the Week 4 library tests and shows the contract tests by name.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings",
     "tests/test_models.py", "tests/test_training.py", "tests/test_datasets.py"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
lines = result.stdout.strip().splitlines()
print("\n".join(line for line in lines if "test_contract_fit_returns_self" in line))
print(lines[-1])
assert result.returncode == 0, result.stdout[-3000:]

**Exercise 4.1.** Write `test_median_baseline_ignores_outlier`: fit your `MedianBaseline` on
the targets `[200, 210, 220, 5000]` (a flow-meter glitch) and assert that it predicts 215 for any
input, while `MeanBaseline` predicts 1407.5. Use `pytest.approx`. In the notebook we call the
function directly; in Practice 4 the same kind of test goes into a `test_*.py` file.

In [ ]:
import pytest


def test_median_baseline_ignores_outlier() -> None:
    """The median is robust to one wild value; the mean is not."""
    X = np.zeros((4, 1))
    y = np.array([200.0, 210.0, 220.0, 5000.0])
    raise NotImplementedError   # TODO: two asserts with pytest.approx (215 and 1407.5)


try:
    test_median_baseline_ignores_outlier()
    print("[x] Exercise 4.1 test passes")
except NotImplementedError:
    print("[ ] Exercise 4.1: not done yet")

## 5. Putting it together: a configuration-driven experiment

With the three patterns in place, an experiment is *data*, not code: a list of dataset names
and model settings. A second small registry maps model names to classes (a factory for
strategies). Nothing below changes when you add a model, a loader or a callback.

In [ ]:
MODEL_REGISTRY: dict[str, type[Regressor]] = {
    cls.name: cls for cls in (MeanBaseline, LinearLeastSquares, KNearestNeighbors, GradientDescentRegressor)
}
CONFIG: list[dict[str, object]] = [
    {"dataset": "fuel_flow_synthetic", "model": "mean_baseline", "params": {}},
    {"dataset": "fuel_flow_synthetic", "model": "linear_least_squares", "params": {"alpha": 1.0}},
    {"dataset": "fuel_flow_synthetic", "model": "k_nearest_neighbors", "params": {"k": 7}},
    {"dataset": "fuel_flow_synthetic", "model": "gradient_descent", "params": {"learning_rate": 0.1}},
]

rows = []
for run in CONFIG:
    data_train, data_test = DatasetFactory.load(str(run["dataset"]), seed=SEED).split()
    strategy = MODEL_REGISTRY[str(run["model"])](**run["params"])   # factory for models
    rows.append(RegressionExperiment(strategy).run(data_train, data_test))  # strategy
summary = pd.DataFrame(rows).set_index("model").round(2)
summary

## 6. Final checks

The cell verifies the worked examples and the library behaviour. It must pass on every machine
(this is the notebook's own test cell); it does not depend on the exercises.

In [ ]:
assert math.isclose(fuel_25, 270.95, abs_tol=0.01)
assert math.isclose(hand_rmse["linear_least_squares"], 14.18, abs_tol=0.01)
assert math.isclose(hand_rmse["k_nearest_neighbors"], 32.60, abs_tol=0.01)
assert math.isclose(gd.weights_[1], 5.0 / 3.0, rel_tol=1e-12)
assert results[-1].model == "KNearestNeighbors(k=5, standardize=False)"   # worse than...
assert results[-2].model == "MeanBaseline()"                # ...the baseline, which all others beat
assert runs[1.0][0] < runs[0.0][0]                         # min_delta stops earlier...
assert abs(runs[1.0][2] - runs[0.0][2]) < 1.0              # ...with practically the same model
assert guard.triggered
assert summary["test_rmse"].idxmin() != "MeanBaseline()"
print("All Week 4 checks passed.")

## What we now know how to do

* Recognise the three axes of change in ML code (model, training behaviour, data source) and the
  pattern that isolates each one.
* **Strategy:** put interchangeable algorithms behind an abstract interface and swap them at run
  time; wrap foreign objects with an **adapter**.
* **Observer:** keep the training loop fixed and add behaviour (history, logging, early stopping,
  divergence guard, schedules) as callbacks that communicate only through a shared state.
* **Factory:** create objects from a name through a registry; extend it with a class decorator.
* Test each pattern with pytest: contract tests over parametrized fixtures, test doubles (spies,
  scripted models), isolation fixtures and `tmp_path`.
* Do one gradient-descent step and an early-stopping trace by hand and check them in code.